# VirtualiZarr → Icechunk for Copernicus Marine Data

This notebook creates a minimal Icechunk store with virtual references to Copernicus Marine Service S3 data,
following the pattern from `pace-chl-icechunk-sc.ipynb`.

**Key Points:**
- Uses HTTPS URLs to Copernicus S3 data (cloudferro endpoint)
- Data stays in original S3 location - no downloading
- Creates virtual references using VirtualiZarr
- Stores in local Icechunk repository

## Dataset
- **Product**: Global Ocean Biogeochemistry L3 Chlorophyll
- **Dataset ID**: `cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D`
- **Endpoint**: `https://s3.waw3-1.cloudferro.com`

In [ ]:
!pip install -qU icechunk virtualizarr copernicusmarine xarray obstore obspec_utils

In [ ]:
import warnings
from pathlib import Path
import xarray as xr
import icechunk
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import HDFParser
from obstore.store import from_url
from obspec_utils.registry import ObjectStoreRegistry

warnings.filterwarnings('ignore', category=UserWarning)

## Step 1: Get S3 URLs from Copernicus Marine Service

Use `copernicusmarine` to get file list without downloading.
This shows the S3 paths which we'll convert to HTTPS URLs.

In [ ]:
# Get file list for a specific date (just one file for minimal example)
!copernicusmarine get \
  --dataset-id cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D \
  --dataset-version 202603 \
  --filter "*20240702*.nc" \
  --create-file-list copernicus_files.txt

In [ ]:
# Read the S3 URLs
with open('copernicus_files.txt', 'r') as f:
    s3_urls = [line.strip() for line in f if line.strip()]

print(f"Found {len(s3_urls)} file(s)")
print(f"S3 URL: {s3_urls[0]}")

## Step 2: Construct HTTPS URLs

Convert S3 URLs to HTTPS format using cloudferro endpoint.
Format: `https://s3.waw3-1.cloudferro.com/bucket/path?x-cop-user=USERNAME`

**Note**: The `?x-cop-user=USERNAME` parameter will be added by users when they access the data.

In [ ]:
# Copernicus cloudferro S3 endpoint
COPERNICUS_ENDPOINT = "https://s3.waw3-1.cloudferro.com"

def s3_to_https(s3_url, endpoint=COPERNICUS_ENDPOINT):
    """Convert s3://bucket/path to https://endpoint/bucket/path"""
    if s3_url.startswith('s3://'):
        path = s3_url[5:]  # Remove 's3://'
        return f"{endpoint}/{path}"
    return s3_url

https_urls = [s3_to_https(url) for url in s3_urls]
test_url = https_urls[0]
print(f"HTTPS URL: {test_url}")
print(f"\nFile name: {Path(test_url).name}")

## Step 3: Set up Remote File Access

Configure how to access the remote Copernicus files (similar to PACE setup).

In [ ]:
# Create object-store handle for the REMOTE files
# For Copernicus cloudferro endpoint
url_prefix = f"{COPERNICUS_ENDPOINT}/"

# Note: For public access testing, we can try without credentials
# In production, users would need to configure Copernicus credentials
store = from_url(url_prefix)
registry = ObjectStoreRegistry({url_prefix: store})
parser = HDFParser()

print(f"✓ Remote storage configured for: {url_prefix}")

## Step 4: Create Virtual Dataset

Open the remote file virtually without downloading data.

In [ ]:
# Open file virtually
# The file is accessed via HTTPS but data is not downloaded
vds = open_virtual_dataset(
    test_url,
    indexes={},
    loadable_variables=['time', 'lat', 'lon', 'latitude', 'longitude'],
    parser=parser,
)

print("✓ Virtual dataset created:")
print(vds)

## Step 5: Set up Icechunk Storage

Create a local Icechunk repository with configuration for virtual chunk access.

In [ ]:
# Set up local storage for Icechunk repository
icechunk_dir = Path("./copernicus_icechunk_minimal")
if icechunk_dir.exists():
    import shutil
    shutil.rmtree(icechunk_dir)
icechunk_dir.mkdir()

storage = icechunk.StorageConfig.filesystem(str(icechunk_dir))

# Configure virtual chunk container
# This tells Icechunk how to access the remote Copernicus data
config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=url_prefix,
        store=icechunk.http_store(),  # Use HTTP store for cloudferro
    )
)

print("✓ Storage configuration created")

In [ ]:
# Create repository
repo = icechunk.Repository.create(storage, config)
session = repo.writable_session(branch="main")

print("✓ Icechunk repository created")

## Step 6: Write Virtual References to Icechunk

In [ ]:
# Write virtual dataset to Icechunk
# This stores references to the remote chunks, not the actual data
vds.virtualize.to_icechunk(session.store)

# Commit the changes
commit_id = session.commit("Initial commit: Copernicus chlorophyll virtual dataset")
print(f"✓ Virtual references written to Icechunk")
print(f"  Commit ID: {commit_id}")

## Step 7: Read from Icechunk Store

Open the Icechunk store and access the data through virtual references.

In [ ]:
# Open repository for reading
repo_read = icechunk.Repository.open(storage, config=config)
session_read = repo_read.readonly_session(branch="main")

# Open with xarray
ds = xr.open_zarr(session_read.store, consolidated=False)

print("✓ Dataset opened from Icechunk store:")
print(ds)

## Note on Data Access

**Important:** The Icechunk store contains virtual references pointing to:
```
https://s3.waw3-1.cloudferro.com/mdl-native-16/native/...
```

To actually **load data** from these URLs, users need Copernicus Marine credentials:

1. Register at https://data.marine.copernicus.eu/register
2. Run `copernicusmarine login` to configure credentials
3. The username will be appended to URLs as `?x-cop-user=USERNAME` when accessing chunks

**For production deployment** (similar to PACE on Source Coop):
- Store Icechunk repository in cloud storage (S3, GCS)
- Configure proper authentication for virtual chunk access
- Users would provide their Copernicus credentials when opening the store

## Comparison with PACE Workflow

| Aspect | PACE OCI | Copernicus Marine |
|--------|----------|-------------------|
| **Data Source** | NASA Earthdata | Copernicus Marine Service |
| **S3 Endpoint** | `s3://ob-cumulus-prod-public/` (AWS us-west-2) | `s3://mdl-native-16/` (cloudferro) |
| **HTTPS Endpoint** | AWS S3 HTTPS | `https://s3.waw3-1.cloudferro.com` |
| **Authentication** | NASA Earthdata (EDL) credentials | Copernicus Marine credentials |
| **URL Discovery** | `earthaccess.search_data()` + `res.data_links()` | `copernicusmarine get --create-file-list` |
| **Virtual Chunk Creds** | `ic.credentials.s3_credentials()` | HTTP store (username in URL param) |

## Summary

This minimal example demonstrates:

1. ✓ Getting S3 URLs from Copernicus Marine Service (no download)
2. ✓ Converting to HTTPS format for cloudferro endpoint
3. ✓ Creating virtual dataset with VirtualiZarr
4. ✓ Configuring virtual chunk container for remote access
5. ✓ Writing virtual references to Icechunk store
6. ✓ Reading dataset from Icechunk store

**Key Achievement:** The Icechunk store contains only metadata and virtual references to the remote Copernicus data. No data was downloaded or duplicated.